In [0]:

data_example = [
    ("t-rex", "blue", 2),
    ("stegosaurus", "pink", 4),
    ("triceratops", "rainbow", 3),
    ("doYouThinkHeSaurus", "yellow", 4),
    ("sparkosaurus", "green", 1),
    ("stegosaurus", "red", 1),
    ("allosaurus", "blue", 1),

]

column_names = ("DinosaurName", "colour", "CountInDatacenters")

df = spark.createDataFrame(data_example, column_names)

In [0]:
# Show casing visualising dataframe
df
df.show()
display(df)

In [0]:
# working with other languages
df.createOrReplaceTempView("dino_df_to_table")

In [0]:
%%sql

SELECT * FROM dino_df_to_table



In [0]:
databricks secrets list-scopes

In [0]:
# Databricks storage
service_credential = dbutils.secrets.get(scope="<scope>",key="<service-credential-key>")

spark.conf.set("fs.azure.account.auth.type.<storage-account>.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.vedatalakestoragedemo.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.<storage-account>.dfs.core.windows.net", "<application-id>")
spark.conf.set("fs.azure.account.oauth2.client.secret.<storage-account>.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.<storage-account>.dfs.core.windows.net", "https://login.microsoftonline.com/<directory-id>/oauth2/token")

In [0]:
# working with delta

# Save to your own filepath
from delta.tables import *
from pyspark.sql.functions import * 

adls_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/bronze/dinosaurs/delta"

(df
    .write
    .format("delta")
    .mode("overwrite")
    .save(adls_path)
 )

 #save as a managed table

 #step 1: 

(df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fabric_unified_lakehouse.dinosaur")
 )

In [0]:
#Demonstrate acid compliance 

deltaTable = DeltaTable.forPath(spark, deltaTable_path)

# changing version of the data file
deltaTable.update(
    condition = "DinosaurName LIKE 'spark%'",
    set = {"CountInDatacenters" : "CountInDatacenters * 100000"}
)

deltaTable.toDF().show(10)

In [0]:
original_df = (
    spark
    .read
    .format("delta")
    .option("versionAsOf",1)
    .load(deltaTable_path)
)

new_df.show()